# Config

In [33]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [34]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json
import numpy as np

# 1) Preprocesamiento de los datos


In [70]:
# 1) Cargar datos
path = "/tmp/data"
path_analytics = "/tmp/analytics"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)
savepath=os.path.join(path_analytics, "deleted_re.xlsx")
df_deleted.to_excel(savepath, index=False)

# 3) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
savepath=os.path.join(path, "data_clean.xlsx")
#df.to_excel(savepath, index=False)

# 2) Traducción del texto

### Original

In [36]:
#Crear columna de registro de idioma: 
# True: Texto en español, False: Texto en inglés
df["Español"]=detect_language(df["Resumen_trad"])

In [112]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()


#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

In [ ]:
#3. Selección de columnas que se utilizarán en clasificador y concatenación
# Última limpieza antes de generar concatenación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad", "Facultad_del_Proyecto_trad", "Depto_Persona_trad"]
for col in cols:
    df[col] = df[col].apply(final_clean)

#  Selección de columnas para embedding.
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names = ["title", "keywords", "abstract"]
df = gen_text_for_embedding(df, cols, element_names)

# Guardado de resultados
savepath=os.path.join(path, "data_translated_concat.xlsx")
df.to_excel(savepath, index=False)
savepath=os.path.join(path, "data_translated_concat.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

df.head()

### Test model

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model = AutoModelForSeq2SeqLM.from_pretrained("google/madlad400-3b-mt")
tokenizer = AutoTokenizer.from_pretrained("google/madlad400-3b-mt")

inputs = tokenizer("<2pt> I love pizza!", return_tensors="pt")
outputs = model.generate(**inputs)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/11.8G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/830 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.43M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.6M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

['Eu amo pizza!']


In [29]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/madlad400-3b-mt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargar modelo y tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

tgt_lang = "en"

text = f"<2{tgt_lang}> PATRONES DE CRIANZA Y SOCIALIZACIÓN DE GÉNERO EN ADOLESCENTES DE FAMILIAS DE CLASES MEDIAS DEL CONCEPCIÓN URBANO".lower()


# Tokenizar y mover a GPU
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(device)

# Generación en GPU
outputs = model.generate(
    **inputs,
    max_new_tokens=512,   # permite traducciones largas
    num_beams=4,          # beam search → mejor calidad
    early_stopping=True
)

# Decodificación
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


parenting patterns and gender socialization in adolescents from middle-class families of urban conception


### New functions

In [68]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import MarianMTModel, MarianTokenizer
import torch
import langid
import re
import pandas as pd
from tqdm import tqdm
import torch

def gen_ids(num, list):
    """
    Repite el ID tantas veces como elementos haya en la lista.
    Args:
        num (int): ID a repetir.
        list (list): Lista cuyos elementos determinan cuántas veces repetir el ID.
    Returns:
        list: Lista con el ID repetido.
    """
    return [num] * len(list)
    
def detect_language(texts):
    """
    Detecta si los textos están en español.
    Args:
        texts (list): Lista de textos a evaluar.
    Returns:
        list: Lista de booleanos indicando si cada texto está en español.
    """
    langs =[]
    for text in texts:
        lang, _ = langid.classify(text)
        if lang == 'es':
            langs.append(True)
        else:
            langs.append(False)
    return langs

class translator():
    """
    Clase para traducir texto del español al inglés utilizando un modelo y tokenizer de Hugging Face.
    Incluye detección de idioma, segmentación en fragmentos y unión de la traducción final.
    La traducción puede realizarse de forma individual, utilizando detect_and_translate(), 
    o bien procesar una lista de textos en paralelo mediante translate_parallel().

    """
    def __init__(self, model_name, model, tokenizer, max_input_tokens=512):
        """
        Inicializa el traductor cargando el modelo y tokenizer en GPU si está disponible.
        Args:
            model: Modelo de traducción.
            tokenizer: Tokenizer asociado al modelo.
            max_input_tokens (int): Máximo de tokens por fragmento.
        """
        if model_name == "google/madlad400-3b-mt":
            self.using_madlad = True
        else:
            self.using_madlad = False

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Usando dispositivo: {self.device}")
        # Cargar modelo y tokenizer
        self.tokenizer = tokenizer
        self.model = model.to(self.device)
        self.max_input_tokens = max_input_tokens

    def split_text(self, text_to_split):
        """
        Divide un texto en fragmentos manejables según el límite de tokens del modelo.
        Args:
            text_to_split (str): Texto original a dividir.
        Returns:
            list: Lista de fragmentos como objetos Document.
        """
        # Splitter basado en el tokenizador de Helsinki (cuenta tokens reales)
        text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            tokenizer=self.tokenizer,
            chunk_size=self.max_input_tokens,
            chunk_overlap=0,
            separators=["\n\n", ".", ",", " "] #Jerarquía de separadores
        )

        texts = text_splitter.create_documents([text_to_split])
        return texts

    def translate_esp_en(self, text_to_split, batch_size=16):
        texts = self.split_text(text_to_split)
        translated_chunks = []

        for i in range(0, len(texts), batch_size):
            batch = [t.page_content for t in texts[i:i+batch_size]]
            encoded = self.tokenizer(batch, return_tensors="pt", padding=True,
                                    truncation=True, max_length=512).to(self.device)
            with torch.inference_mode():
                out_ids = self.model.generate(
                    **encoded,
                    num_beams=4,
                    max_new_tokens=self.max_input_tokens,
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )
            translated_batch = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
            translated_chunks.extend(translated_batch)

        return "\n\n".join(translated_chunks)

    # Detección y traducción de texto. 
    def detect_and_translate(self, text):
        """
        Detecta el idioma del texto y lo traduce si está en español.
        Args:
            text (str): Texto de entrada.
        Returns:
            str: Texto traducido o el mismo texto si no es español.
        """
        lang, _ = langid.classify(text)
        if lang == 'es':
            return self.translate_esp_en(text)
        
        return text

    def translate_parallel(self, texts, batch_size=16):
        """
        Traduce en paralelo una lista de textos mezclados en español e inglés.

        Inputs:
            texts (list[str]): Lista de textos a traducir.

        Outputs:
            list[str]: Lista de textos donde los que estaban en español fueron traducidos
                    y los que estaban en otros idiomas se mantienen igual.

        Proceso:
            1. Detecta qué textos están en español.
            2. Divide los textos largos en fragmentos (para no superar límite de tokens).
            3. Traduce por lotes con el modelo de traducción.
            4. Reconstruye los textos traducidos completos.
            5. Une los textos traducidos con los originales en otros idiomas, manteniendo orden.
        """

        # Crear lista de IDs únicos para no perder el orden
        ids = list(range(len(texts)))

        # Detectar idioma de cada texto (True = español, False = otro idioma)
        langs = detect_language(texts)
   

        # Construir dataframe base
        df = pd.DataFrame({'id': ids, 'is_spanish': langs, 'text': texts})

        # Separar en textos español e inglés
        df_spanish = df[df['is_spanish']].copy()
        df_other   = df[~df['is_spanish']].copy()

        # Dividir textos españoles en fragmentos manejables
        df_spanish['text'] = df_spanish['text'].apply(self.split_text)

        # Generar lista expandida de (id, fragmento)
        id_loc, texts_for_batch = [], []
        for _, row in df_spanish.iterrows():
            id_loc.extend(gen_ids(row['id'], row['text']))   # genera ID por fragmento
            texts_for_batch.extend(row['text'])              # agrega fragmentos
        
        #Añadir token de idioma si es MadLad
        if self.using_madlad:
            print("Usando MadLad para traducción por lotes")
            for i in range(len(texts_for_batch)):
                texts_for_batch[i] = f"<2en> {texts_for_batch[i]}"

        # === Traducción por lotes ===
        translated_chunks = []
        
        for i in tqdm(range(0, len(texts_for_batch), batch_size), desc="Traduciendo", unit="batch"):
            #batch = [t.page_content for t in texts_for_batch[i:i+batch_size]]
            batch = [
                t.page_content if hasattr(t, "page_content") else t
                for t in texts_for_batch[i:i+batch_size]
            ]
            # Tokenizar batch
            encoded = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(self.device)

            # Generación de traducción
            with torch.inference_mode():
                out_ids = self.model.generate(
                    **encoded,
                    num_beams=4,
                    max_new_tokens=self.max_input_tokens,
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )
            # Decodificar traducciones y acumular
            translated_batch = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
            translated_chunks.extend(translated_batch)

        # === Reconstruir textos ===
        df_translated = pd.DataFrame({'id': id_loc, 'text': translated_chunks})
        df_translated = (
            df_translated.groupby('id')['text']
            .apply(lambda x: ' '.join(x))      # unir fragmentos del mismo texto
            .reset_index()
        )

        # Unir con textos originales en otros idiomas y ordenar
        df_final = pd.concat(
            [df_translated, df_other.drop(columns=['is_spanish'])],
            ignore_index=True
        ).sort_values(by="id").reset_index(drop=True)

        return df_final["text"].to_list()
    
def gen_text_for_embedding(df, cols, element_names=None, sep=" "):
    """
    df            : DataFrame de entrada
    cols          : lista de nombres de columnas a procesar y concatenar
    element_names : lista de etiquetas para cada columna (misma longitud que cols)
    sep           : separador entre pares clave-valor (default = " ")
    """
    if element_names is None:
        df["text_for_embedding_translated"] = df[cols].agg(sep.join, axis=1)
    else:
        if len(element_names) != len(cols):
            raise ValueError("element_names debe tener la misma longitud que cols")

        df["text_for_embedding_translated"] = df[cols].agg(
            lambda row: sep.join(
                f"{name}: {row[col]}" for col, name in zip(cols, element_names)
            ),
            axis=1
        )
    return df

def final_clean(text):
    """
    Limpia un texto eliminando saltos de línea, tabs y espacios múltiples.
    Args:
        text (str): Texto de entrada.
    Returns:
        str: Texto limpio y sin espacios innecesarios.
    """
    if not isinstance(text, str):
        return ""
    # Reemplaza saltos de línea y tabs por un espacio
    text = re.sub(r'[\r\n\t]+', ' ', text)
    # Colapsa espacios múltiples
    text = re.sub(r'\s+', ' ', text)
    #Lleva todo a minúscula
    text = text.lower()
    
    return text.strip()

### Functional

In [71]:

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/madlad400-3b-mt"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
trans = translator(model_name, model, tokenizer)
"""
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model_name, model, tokenizer)
"""
#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
df=df.iloc[0:1] 
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()


#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

print(f"Tiempo total de traducción: {end - start:.2f} segundos")


Usando dispositivo: cuda
Usando MadLad para traducción por lotes


Traduciendo: 100%|██████████| 1/1 [00:01<00:00,  1.03s/batch]


Usando MadLad para traducción por lotes


Traduciendo: 100%|██████████| 1/1 [00:26<00:00, 26.69s/batch]


Usando MadLad para traducción por lotes


...

Usando MadLad para traducción por lotes


Traduciendo: 0batch [00:00, ?batch/s]
/tmp/ipykernel_1261/1869570980.py:204: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda x: ' '.join(x))      # unir fragmentos del mismo texto


Usando MadLad para traducción por lotes


Traduciendo: 100%|██████████| 1/1 [00:01<00:00,  1.25s/batch]

Tiempo total de traducción: 48.39 segundos


In [75]:
df.columns

Index(['Código VRID', 'Interdisciplinario', 'Transdisciplinario', 'Título',
       'Keywords', 'Resumen', 'Facultad del Proyecto', 'Depto Persona',
       'Titulo_trad', 'Resumen_trad', 'keywords_trad',
       'Facultad_del_Proyecto_trad', 'Depto_Persona_trad'],
      dtype='object')

In [81]:
df["Keywords"].iloc[0]

'ECONOMÍA AMBIENTAL, ECONOMÍA DE RECURSOS NATURALES Y DEL MEDIO AMBIENTE, _x000D_\nDESARROLLO ECONÓMICO, TRANSICIONES ENERGÉTICAS SUSTENTABLES, GESTIÓN DE RESIDUOS, PESCA Y _x000D_\nACUICULTURA, COMPORTAMIENTO PRO-AMBIENTAL, ECONOMÍA DEL COMPORTAMIENTO, MICROECONOMÍA _x000D_\nAPLICADA.'

# 3) Split dataset

In [11]:
def to_serializable(obj):
    if hasattr(obj, "tolist"):
        return obj.tolist()
    return obj

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
import json

path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

df = df[df["Interdisciplinario"] != "INDEFINIDO"]
le = LabelEncoder()
df["labels"] = le.fit_transform(df["Interdisciplinario"])

# Convertir a arrays
ids = df["Código VRID"].to_numpy()
labels = df["labels"].to_numpy()

# Train/Test split (ids y labels en paralelo)
idx_train, idx_test, y_train, y_test = train_test_split(
    ids,
    labels,
    test_size=0.2,
    random_state=7,
    stratify=labels
)

# Crear folds sobre train
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

folds = []
for fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):
    train_ids = idx_train[train_pos]   # array de IDs
    val_ids = idx_train[val_pos]       # array de IDs
    folds.append(val_ids)

print("Test size:", len(idx_test))
print("Fold 0 - Val size:", len(folds[0]))

#Guardar index en diccionario
dataset_index = {
    "Train": idx_train,
    "Test": idx_test,
    "kfolds": folds 
}
filepath=os.path.join(path, "train_test_ids_3folds.json")

# Guardar
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)


Test size: 193
Fold 0 - Val size: 257


# 4) TF-ID feature extractor 

## Train

In [115]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Prueba con solo textos traducidos
#df = df[df["Español"]==False]

In [116]:
from sklearn.preprocessing import LabelEncoder
from models.TIFD import gen_TFID_vectors
from utils.dataset import gen_dataset
import numpy as np

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


(771, 16724) (193, 16724)


In [117]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.63, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.63, 'std_test_score': 0.02}


In [118]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

## Save

In [ ]:
exp_info = {
    'exp_name': "Bayesiansearchcv_TFID_f1w",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/08/29 19:51:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/2/runs/64dda3e462a84ff1a5d8f8c65a1dc718
🧪 View experiment at: http://mlflow-server:5000/#/experiments/2
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/08/29 19:51:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/2/runs/fbe45024b39a455f8e445de5d9f04f59
🧪 View experiment at: http://mlflow-server:5000/#/experiments/2
📝 Registrando modelo en MLflow: XGBClassifier


2025/08/29 19:51:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/2/runs/93522d0f7e294637a946b5a139c368e9
🧪 View experiment at: http://mlflow-server:5000/#/experiments/2
📝 Registrando modelo en MLflow: SVC


2025/08/29 19:52:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/2/runs/1a7920463a51473ca55d4c2ef5d777c0
🧪 View experiment at: http://mlflow-server:5000/#/experiments/2


# 5) SPECTER model

## Train

In [3]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [4]:
#del gen_dataset
from utils.dataset import gen_dataset
from models.specter import embed_texts
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# 2) Calcular embeddings
# Parámetros modelo
BASE_MODEL = "allenai/specter2_base"
#ADAPTER_NAME = "allenai/specter2"
ADAPTER_NAME="allenai/specter2_classification"
X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)

print(X_train.shape, X_test.shape)

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:830: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(771, 768) (193, 768)


In [5]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.64, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.01}
XGBClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.63, 'std_test_score': 0.01}


In [6]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6476683937823834, 'precision': 0.7087378640776699, 'recall': 0.6576576576576577, 'f1_macro': 0.6434470767224516, 'cm': array([[52, 30],
       [38, 73]]), 'f1_es': 0.6325030804435839, 'f1_en': 0.6575154426904598, 'cm_es': array([[18, 22],
       [20, 55]]), 'cm_en': array([[34,  8],
       [18, 18]])}
RandomForestClassifier
{'accuracy': 0.6683937823834197, 'precision': 0.7079646017699115, 'recall': 0.7207207207207207, 'f1_macro': 0.6596119929453264, 'cm': array([[49, 33],
       [31, 80]]), 'f1_es': 0.6373445535296867, 'f1_en': 0.7038019451812555, 'cm_es': array([[17, 23],
       [18, 57]]), 'cm_en': array([[32, 10],
       [13, 23]])}
XGBClassifier
{'accuracy': 0.6839378238341969, 'precision': 0.7049180327868853, 'recall': 0.7747747747747747, 'f1_macro': 0.6697523072175937, 'cm': array([[46, 36],
       [25, 86]]), 'f1_es': 0.6669747772937873, 'f1_en': 0.6927133512499366, 'cm_es': array([[17, 23],
       [14, 61]]), 'cm_en': array([[29, 13],
       [1

## Save

In [ ]:
exp_info = {
    'exp_name': "Bayesiansearchcv_specter_f1m",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/08/29 20:00:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/1/runs/59d9bf17d81740c9808412615b4f1223
🧪 View experiment at: http://mlflow-server:5000/#/experiments/1
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/08/29 20:00:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/1/runs/43ab81c635d84877871f6c191ab03d9d
🧪 View experiment at: http://mlflow-server:5000/#/experiments/1
📝 Registrando modelo en MLflow: XGBClassifier


2025/08/29 20:00:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/1/runs/3b4c012864f34c639fdfa554a4fce910
🧪 View experiment at: http://mlflow-server:5000/#/experiments/1
📝 Registrando modelo en MLflow: SVC


2025/08/29 20:00:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/1/runs/a4ecef4973ba4ca0bb6d6fa338bb7e92
🧪 View experiment at: http://mlflow-server:5000/#/experiments/1


# 6) ROBERTA

In [4]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [8]:
#del gen_dataset
from utils.dataset import gen_dataset
from models.specter import embed_texts
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
from transformers import RobertaTokenizerFast, RobertaModel
import torch
import warnings
from tqdm import tqdm
warnings.filterwarnings("ignore", message="Some weights of the model.*were not initialized.*")

def roberta_encoder_batch(texts, batch_size=8, max_length=512):
    model_name = "roberta-large"
    tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
    model = RobertaModel.from_pretrained(model_name)

    model.eval()  # desactiva dropout
    embeddings = []

    # recorrer en lotes de batch_size
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        # tokenización por lote
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        with torch.no_grad():
            outputs = model(**inputs)

        # embeddings del token <s> ([CLS]) para cada texto del batch
        cls_embeddings = outputs.last_hidden_state[:, 0, :]  # (batch, hidden_dim)
        embeddings.append(cls_embeddings.cpu().numpy())

    # concatenar todos los batches
    return np.vstack(embeddings)  # (n_texts, hidden_dim)

def roberta_encoder(texts):
    embeddings = []
    model_name = "roberta-large"
    tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
    model = RobertaModel.from_pretrained(model_name)
    
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)

        with torch.no_grad():
            outputs = model(**inputs)

        # outputs.last_hidden_state → (batch_size, seq_len, hidden_dim)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # primer token (<s>), equivalente a [CLS]
        embeddings.append(cls_embedding)
    
    return np.array(embeddings)



In [ ]:
X_train = roberta_encoder_batch(X_train)
X_test = roberta_encoder_batch(X_test)

In [35]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

Exception ignored in: <function tqdm.__del__ at 0x7fb522506680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/tqdm/std.py", line 1149, in __del__
    self.close()
  File "/usr/local/lib/python3.10/dist-packages/tqdm/std.py", line 1278, in close
    if self.last_print_t < self.start_t + self.delay:
AttributeError: 'tqdm' object has no attribute 'last_print_t'


📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.61, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.6, 'std_test_score': 0.04}
XGBClassifier: {'mean_test_score': 0.6, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.6, 'std_test_score': 0.03}


In [36]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6683937823834197, 'precision': 0.7043478260869566, 'recall': 0.7297297297297297, 'f1_macro': 0.6584070796460177, 'cm': array([[48, 34],
       [30, 81]]), 'f1_es': 0.645422630299757, 'f1_en': 0.6745562130177515, 'cm_es': array([[15, 25],
       [14, 61]]), 'cm_en': array([[33,  9],
       [16, 20]])}
RandomForestClassifier
{'accuracy': 0.6632124352331606, 'precision': 0.6575342465753424, 'recall': 0.8648648648648649, 'f1_macro': 0.6216028715350044, 'cm': array([[32, 50],
       [15, 96]]), 'f1_es': 0.5826351679273679, 'f1_en': 0.6671052631578948, 'cm_es': array([[ 5, 35],
       [ 4, 71]]), 'cm_en': array([[27, 15],
       [11, 25]])}
XGBClassifier
{'accuracy': 0.6373056994818653, 'precision': 0.6722689075630253, 'recall': 0.7207207207207207, 'f1_macro': 0.6234671125975473, 'cm': array([[43, 39],
       [31, 80]]), 'f1_es': 0.6272391754433342, 'f1_en': 0.6265328874024525, 'cm_es': array([[14, 26],
       [15, 60]]), 'cm_en': array([[29, 13],
       [16